In [ ]:
from Bio import Entrez
import xml.etree.ElementTree as ET
import pandas as pd
from collections import defaultdict
from typing import Dict, List
import re

In [ ]:
Entrez.email = "lukas.becker@hhu.de"

In [ ]:
def get_srr_data(uids) -> str:
    xml = Entrez.efetch(db="sra", id=uids, retmode="xml").read()
    return xml

In [ ]:
bioproject_to_srr_uids = pd.read_table("../data/bioproject_srr_table.tsv", sep="\t")
bioproject_to_srr_uids.head()

In [ ]:
bioproject524291 = bioproject_to_srr_uids[bioproject_to_srr_uids["BioProject"] == 524291]
bioproject524291.head()

In [ ]:
xmls = get_srr_data(list(bioproject524291.SRR))

In [ ]:
xmls

In [ ]:
root = ET.fromstring(xmls)

packages = root.findall(".//EXPERIMENT_PACKAGE")
print(f"Found {len(packages)} experiment packages")

srr_file_to_sample_description = {}
for i, pkg in enumerate(packages, 1):
    title = pkg.findtext(".//TITLE")
    organism = pkg.findtext(".//SCIENTIFIC_NAME")
    run_accessions = [
        run.attrib.get("accession")
        for run in pkg.findall(".//RUN")
    ]

    srr_file_to_sample_description[run_accessions[0]] = title
    #print(f"\nEntry {i}")
    #print("  Title:", title)
    #print("  Organism:", organism)
    #print("  Runs:", run_accessions)

In [ ]:
srr_file_to_sample_description

In [ ]:
def group_by_strain_and_temp(srr_to_name: Dict[str, str]) -> Dict[str, List[str]]:
    """
    Groups SRRs by (strain, temperature) using value strings like:
      CC7_39_t25_5_R1  -> group key: "CC7_t25"
      H2_42_t34_6_R2   -> group key: "H2_t34"

    Non-matching names (e.g., neg_PCR_control_1, WS_39) are grouped under:
      "controls_neg_PCR_control" or "WS"
    """
    grouped = defaultdict(list)

    # Pattern: strain_time_temp_rep(_R1/_R2 optional)
    # strain: CC7 or H2
    # temp: t25 or t34 (but this will capture any t\d+)
    pat = re.compile(r'^(?P<strain>CC7|H2)_(?:\d+?)_(?P<temp>t\d+)(?:_.*)?$')

    for srr, name in srr_to_name.items():
        m = pat.match(name)
        if m:
            key = f"{m.group('strain')}_{m.group('temp')}"   # e.g., CC7_t25
        else:
            if name.startswith("neg_PCR_control"):
                key = "controls_neg_PCR_control"
            elif name.startswith("WS_"):
                key = "WS"
            else:
                key = "other"

        grouped[key].append(srr)

    # Optional: stable ordering
    return {k: sorted(v) for k, v in grouped.items()}

# ---- example usage ----
grouped = group_by_strain_and_temp(srr_file_to_sample_description)

for k, v in grouped.items():
    print(k, len(v))

In [ ]:
with open("../results/sample-metadata.tsv","w") as f:
    f.write("sample-id\ttreatment\n")
    for experiment in grouped.keys():
        for sample in grouped[experiment]:
            f.write("{}\t{}\n".format(sample, experiment))

In [ ]:
bioproject576020 = bioproject_to_srr_uids[bioproject_to_srr_uids["BioProject"] == 576020]
bioproject576020.head()

In [ ]:
xmls = get_srr_data(list(bioproject576020.SRR))

In [ ]:
xmls

In [ ]:
xmls

In [ ]:
root = ET.fromstring(xmls)

packages = root.findall(".//EXPERIMENT_PACKAGE")
print(f"Found {len(packages)} experiment packages")

srr_file_to_sample_description = {}
for i, pkg in enumerate(packages, 1):
    title = pkg.findtext(".//LIBRARY_NAME")
    organism = pkg.findtext(".//DESIGN_DESCRIPTION")
    # print(organism)
    run_accessions = [
        run.attrib.get("accession")
        for run in pkg.findall(".//RUN")
    ]

    srr_file_to_sample_description[run_accessions[0]] = title, organism
    #print(f"\nEntry {i}")
    #print("  Title:", title)
    #print("  Organism:", organism)
    #print("  Runs:", run_accessions)

In [ ]:
srr_file_to_sample_description

In [ ]:
import re
from collections import defaultdict

def group_srrs(d, cultures_meta_pat=r"^AIMS[1-4]_Anemone_Control$", water_meta_pat=r"^AIMS[1-4]_SW_Control$"):
    items = [{"srr": srr, "name": name, "meta": meta} for srr, (name, meta) in d.items()]

    def tank_id(name, prefix):
        m = re.match(rf"^({prefix}\d+)-", name)
        return m.group(1) if m else None

    groups = defaultdict(list)

    # Cultures: keep all replicates (6 tanks * 3 reps * 4 genotypes = 72)
    for it in items:
        if re.match(cultures_meta_pat, it["meta"]):
            groups["cultures"].append(it)

    # Water: one per tank (1 L per tank * 3 tanks * 4 genotypes = 12)
    water = [it for it in items if re.match(water_meta_pat, it["meta"])]
    water_by_tank = defaultdict(list)
    for it in water:
        tid = tank_id(it["name"], "AN")
        if tid:
            water_by_tank[tid].append(it)
    # pick first SRR per tank (sorted for stability)
    for tid, lst in water_by_tank.items():
        groups["water"].append(sorted(lst, key=lambda x: x["srr"])[0])

    # DNA extraction blanks: pick 1 tissue + 1 water
    tissue_blanks = [it for it in items if re.match(r"^ExtBlank\.", it["name"])]
    water_blanks = [it for it in items if re.match(r"^Ext\.Blank\d+_FWD", it["name"])]
    if tissue_blanks:
        groups["blank_tissue"].append(sorted(tissue_blanks, key=lambda x: x["srr"])[0])
    if water_blanks:
        groups["blank_water"].append(sorted(water_blanks, key=lambda x: x["srr"])[0])

    # NTPCR + Artemia
    groups["ntpcr"] = [it for it in items if re.match(r"^NTPCR\.", it["name"])]
    groups["artemia"] = [it for it in items if re.match(r"^Artemia\.", it["name"])]

    return groups

# --- run ---
groups = group_srrs(srr_file_to_sample_description)

# --- quick summary ---
print("Cultures SRRs:", len(groups["cultures"]))
print("Water SRRs:", len(groups["water"]))
print("Blank tissue:", groups["blank_tissue"])
print("Blank water:", groups["blank_water"])
print("NTPCR SRRs:", [x["srr"] for x in groups["ntpcr"]])
print("Artemia SRRs:", [x["srr"] for x in groups["artemia"]])